In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import time

# ==========================================
# Task 1: The Standardized Pipelines
# ==========================================
class ConvertRGB:
    def __call__(self, img):
        return img.convert('RGB')

# Standard Pipeline (224x224)
transform_224 = transforms.Compose([
    ConvertRGB(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Inception Pipeline (299x299)
transform_299 = transforms.Compose([
    ConvertRGB(),
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Preparing Caltech-101 Datasets...")
# We must load the dataset twice to apply the different transforms
dataset_224 = datasets.Caltech101(root='./data', download=True, transform=transform_224)
dataset_299 = datasets.Caltech101(root='./data', download=True, transform=transform_299)

train_size = int(0.8 * len(dataset_224))
test_size = len(dataset_224) - train_size

# Use the EXACT same random seed for both splits so the images stay consistent
generator = torch.Generator().manual_seed(42)

train_224, test_224 = random_split(dataset_224, [train_size, test_size], generator=generator)
train_299, test_299 = random_split(dataset_299, [train_size, test_size], generator=generator)

# Standard Loaders
train_loader_224 = DataLoader(train_224, batch_size=32, shuffle=True)
test_loader_224 = DataLoader(test_224, batch_size=32, shuffle=False)

# Inception Loaders
train_loader_299 = DataLoader(train_299, batch_size=32, shuffle=True)
test_loader_299 = DataLoader(test_299, batch_size=32, shuffle=False)


# ==========================================
# Task 2: The Contenders
# ==========================================
class BaselineFCN(nn.Module):
    def __init__(self, num_classes=101):
        super(BaselineFCN, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(3 * 224 * 224, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

class AdaptedLeNet5(nn.Module):
    def __init__(self, num_classes=101):
        super(AdaptedLeNet5, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, kernel_size=5) 
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) 
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5) 
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) 
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(16 * 53 * 53, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, num_classes)

    def forward(self, x):
        x = self.pool1(torch.relu(self.conv1(x)))
        x = self.pool2(torch.relu(self.conv2(x)))
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

def get_pretrained_model(model_name, num_classes=101):
    if model_name == 'alexnet':
        model = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif model_name == 'vgg16':
        model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif model_name == 'resnet18':
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == 'inception_v3':
        model = models.inception_v3(weights=models.Inception_V3_Weights.DEFAULT)
        # INSTRUCTOR NOTE: Disable aux_logits to prevent the model from returning a tuple during training
        model.aux_logits = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == 'mobilenet_v2':
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif model_name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

# ==========================================
# Task 3: The Benchmark Protocol
# ==========================================
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def benchmark_model(model, name, t_loader, v_loader, epochs=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    num_params = count_parameters(model)
    print(f"\n--- Benchmarking {name} ---")
    
    start_time = time.time()
    for epoch in range(epochs):
        model.train()
        for images, labels in t_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
    training_time = time.time() - start_time
    time_per_epoch = training_time / epochs
    
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in v_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    val_acc = 100 * correct / total
    
    return {"Name": name, "Params": num_params, "Time/Epoch": time_per_epoch, "Accuracy": val_acc}

# Execution List (Tuples of: Name, Model, TrainLoader, TestLoader)
run_queue = [
    ("FCN Baseline", BaselineFCN(), train_loader_224, test_loader_224),
    ("LeNet-5", AdaptedLeNet5(), train_loader_224, test_loader_224),
    ("AlexNet", get_pretrained_model('alexnet'), train_loader_224, test_loader_224),
    ("VGG-16", get_pretrained_model('vgg16'), train_loader_224, test_loader_224),
    ("ResNet-18", get_pretrained_model('resnet18'), train_loader_224, test_loader_224),
    # Note the 299 loaders passed to Inception!
    ("Inception-v3", get_pretrained_model('inception_v3'), train_loader_299, test_loader_299),
    ("MobileNetV2", get_pretrained_model('mobilenet_v2'), train_loader_224, test_loader_224),
    ("EfficientNet-B0", get_pretrained_model('efficientnet_b0'), train_loader_224, test_loader_224)
]
# ==========================================
# Task 4: The Final Report
# ==========================================
results = []
for name, net, t_loader, v_loader in run_queue:
    res = benchmark_model(net, name, t_loader, v_loader)
    results.append(res)

print("\n" + "="*70)
print(f"{'Model Architecture':<20} | {'Parameters':<12} | {'Time/Epoch (s)':<15} | {'Accuracy'}")
print("="*70)
for r in results:
    print(f"{r['Name']:<20} | {r['Params']:<12,} | {r['Time/Epoch']:<15.2f} | {r['Accuracy']:.2f}%")
print("="*70)

"""
Instructor Notes / Analytical Answers:

1. FCN Performance: It flattens the image immediately, destroying 2D spatial relationships. 
   Pixels are treated as independent features rather than parts of shapes or edges.

2. VGG-16 vs ResNet-18: VGG-16 uses massive dense layers at the end and simply stacks 
   convolutions. ResNet introduced "Skip Connections" (Residual blocks), allowing gradients 
   to flow past dead layers. This solved the vanishing gradient problem, allowing the network 
   to be deeper but much more mathematically efficient, drastically reducing parameters 
   while improving feature extraction.

3. EfficientNet-B0 vs Inception-v3: Inception uses complex branching blocks (parallel 
   convolutions of different sizes). EfficientNet uses a streamlined architecture with 
   Depthwise Separable Convolutions and compound scaling, meaning it executes fewer, cheaper 
   floating-point operations to achieve the same or better representational power.
"""